<a href="https://colab.research.google.com/github/Murilo1514/IA025-UNICAMP/blob/main/(2026S2)_Aula_8_BERT_classificador_de_trechos_de_livros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 8 - BERT classificador de trechos de livros

Nome: Murilo Batista da Silva | RA:276821


## Instruções:

Criar um classificar de trechos do dataset Machado de Assis como fazendo parte do livro correto (dentre 8 livros), utilizando Embeddings gerados por um BERT pré-treinado e uma MLP no token [CLS].

Objetivos:
- O modelo deve associar um trecho de livro ao livro correspondente (classificação).
- A modelagem do dataset, criação de chunks de texto, limpeza dos dados, split, é sua responsabilidade.

Detalhes:
- Deve-se implementar o próprio laço de treinamento. **Não usar frameworks de treinamento automático.**
- Utilizar o BertModel.from_pretrained e BertTokenizer.from_pretrained do Hugging Face para carregar o BERT pré-treinado.
- Não utilize outras classes da HuggingFace/Transformers alem da [BertModel](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertModel) e [BertTokenizer](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertTokenizer).
- O BERT original do artigo pode ser difícil de rodar, tentar utilizar variações com menos parâmetros, ou pré-treinadas em PT-BR (como BERTimbau).
- Deixe para usar GPU somente quando tudo estiver validado.

Tarefas OPCIONAIS, para quem tem computação:
- Experimente com mudar o contexto e congelar ou não os parâmetros do BERT. Cuidado que o contexto é o maior fator para o peso computacional aqui.                                                                             
- A classificação usando livros_com_nomes deve ser mais fácil do que usando os livros_sem_nomes. Compare os resultados entre usar livros_com_nomes e livros_sem_nomes.           

Além do notebook com todas as saídas e com métricas de classificação, entregue uma breve discussão do que foi feito.
                                                                             

In [1]:
import random
import torch
import torch.nn.functional as F
import numpy as np

In [2]:
random.seed(123)
np.random.seed(123)
torch.manual_seed(123)

## Estudo BERT

### Tokenizer
-Testar com strings\
-Tokens especiais\
-Tokenize\
-Decode\
-Batch e padding


In [3]:
from transformers import BertTokenizer

In [4]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
# tokenização
tokenizer("Hello World").input_ids

[101, 7592, 2088, 102]

In [8]:
# tokenização
tokenizer("Hello World")

{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [9]:
# tokens especiais
tokenizer.decode([0, 101, 102, 103])

'[PAD] [CLS] [SEP] [MASK]'

In [42]:
tokenizer("Hello World Orthogonalization")

{'input_ids': [101, 7592, 2088, 28721, 3989, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [43]:
# palavras são quebradas em subwords
tokenizer.tokenize("Hello World Orthogonalization")

['hello', 'world', 'orthogonal', '##ization']

In [44]:
tokenizer.tokenize("Testando uma frase em português!")

['test',
 '##ando',
 'um',
 '##a',
 'fra',
 '##se',
 'em',
 'port',
 '##ug',
 '##ues',
 '!']

In [45]:
input_str = "Machado de Assis é um escritor famoso."
tokenizer.tokenize(input_str)

['mach',
 '##ado',
 'de',
 'ass',
 '##is',
 'e',
 'um',
 'es',
 '##cr',
 '##itor',
 'fa',
 '##mos',
 '##o',
 '.']

In [46]:
tokenizer(input_str)

{'input_ids': [101, 24532, 9365, 2139, 4632, 2483, 1041, 8529, 9686, 26775, 15660, 6904, 15530, 2080, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [47]:
# tokenizador ajustado para PT-BR
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [48]:
tokenizer(input_str)

{'input_ids': [101, 7445, 125, 11160, 253, 222, 4223, 4828, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [49]:
tokenizer.tokenize(input_str)

['Machado', 'de', 'Assis', 'é', 'um', 'escritor', 'famoso', '.']

In [52]:
tokenizer.tokenize("Repara nessa frase")

['Re', '##para', 'nessa', 'frase']

In [53]:
t = tokenizer(["Primeira sentença essa é muito maior",
               "Segunda sentença"], max_length=16, padding="max_length", truncation=False)
t.input_ids, t.attention_mask

([[101, 4085, 12385, 1921, 253, 785, 636, 102, 0, 0, 0, 0, 0, 0, 0, 0],
  [101, 3335, 12385, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
 [[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
  [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [60]:
for x in t.input_ids:
    print(tokenizer.decode(x))

[CLS] Primeira sentença essa é muito maior [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] Segunda sentença [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [63]:
tokenizer.model_max_length

1000000000000000019884624838656

## Como attention mask funciona?

In [64]:
Q = torch.tensor([[0, 1], [0, 1], [3, 3]]).float()
K = Q
V = Q

In [65]:
Q, K, V

(tensor([[0., 1.],
         [0., 1.],
         [3., 3.]]),
 tensor([[0., 1.],
         [0., 1.],
         [3., 3.]]),
 tensor([[0., 1.],
         [0., 1.],
         [3., 3.]]))

In [66]:
K.transpose(1, 0)

tensor([[0., 0., 3.],
        [1., 1., 3.]])

In [67]:
# Primeiro passo da atenção: relação NxN entre embeddings entre Q e K através de QK^T
scores = Q @ K.transpose(1, 0)
scores

tensor([[ 1.,  1.,  3.],
        [ 1.,  1.,  3.],
        [ 3.,  3., 18.]])

In [77]:
scores[:, 2] = -10**7
scores

tensor([[ 1.0000e+00,  1.0000e+00, -1.0000e+07],
        [ 1.0000e+00,  1.0000e+00, -1.0000e+07],
        [ 3.0000e+00,  3.0000e+00, -1.0000e+07]])

In [78]:
# Transformando em scores
attention = torch.nn.Softmax(dim=1)(scores)
attention

tensor([[0.5000, 0.5000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.5000, 0.5000, 0.0000]])

In [81]:
V

tensor([[0., 1.],
        [0., 1.],
        [3., 3.]])

In [80]:
# Scores ponderam V
attention @ V

tensor([[0., 1.],
        [0., 1.],
        [0., 1.]])

In [ ]:
# PERGUNTA: como eliminar a influência do embedding 3,3 (terceiro token)?

### Forward no modelo
Inspecionar modelo com torchinfo\
Visualizar saida do modelo

In [82]:
!pip install torchinfo -q

In [83]:
import torchinfo
from transformers import BertModel

In [84]:
model = BertModel.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [85]:
test_input = tokenizer("Hello BERT!", return_tensors='pt').input_ids
test_input, test_input.shape

(tensor([[  101, 15044, 22280,   241,  5054, 22321,   106,   102]]),
 torch.Size([1, 8]))

In [86]:
output = model(input_ids=test_input)

In [87]:
list(output.keys())

['last_hidden_state', 'pooler_output']

In [88]:
# "estado final dos embeddings"
output.last_hidden_state.shape

output.last_hidden_state[:, 0, :].shape

torch.Size([1, 768])

In [89]:
# O que esta dentro da classe?
torchinfo.summary(model,
                  input_data=test_input,
                  depth=4)

Layer (type:depth-idx)                             Output Shape              Param #
BertModel                                          [1, 768]                  --
├─BertEmbeddings: 1-1                              [1, 8, 768]               --
│    └─Embedding: 2-1                              [1, 8, 768]               23,440,896
│    └─Embedding: 2-2                              [1, 8, 768]               1,536
│    └─Embedding: 2-3                              [1, 8, 768]               393,216
│    └─LayerNorm: 2-4                              [1, 8, 768]               1,536
│    └─Dropout: 2-5                                [1, 8, 768]               --
├─BertEncoder: 1-2                                 [1, 8, 768]               --
│    └─ModuleList: 2-6                             --                        --
│    │    └─BertLayer: 3-1                         [1, 8, 768]               --
│    │    │    └─BertAttention: 4-1                [1, 8, 768]               2,363,904
│    │   

In [98]:
# O BERT TINY é a mesma coisa, só escalado para muito menos camadas!
model = BertModel.from_pretrained('prajjwal1/bert-tiny')

torchinfo.summary(model,
                  input_data=test_input,
                  depth=3)

config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 17.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layer (type:depth-idx)                             Output Shape              Param #
BertModel                                          [1, 128]                  --
├─BertEmbeddings: 1-1                              [1, 8, 128]               --
│    └─Embedding: 2-1                              [1, 8, 128]               3,906,816
│    └─Embedding: 2-2                              [1, 8, 128]               256
│    └─Embedding: 2-3                              [1, 8, 128]               65,536
│    └─LayerNorm: 2-4                              [1, 8, 128]               256
│    └─Dropout: 2-5                                [1, 8, 128]               --
├─BertEncoder: 1-2                                 [1, 8, 128]               --
│    └─ModuleList: 2-6                             --                        --
│    │    └─BertLayer: 3-1                         [1, 8, 128]               198,272
│    │    └─BertLayer: 3-2                         [1, 8, 128]               198,272
├─BertPooler

## Dados NOVOS: machado_livros

Vamos usar UMA NOVA VERSÃO do dataset do Machado de Assis, dividido nos livros principais do autor.

Existem dois conjuntos: um com anonimização dos nomes e outro sem. A pasta livros_com_nomes deve ser uma fonte de dados mais fácil de classificar do que livros_sem_nomes.


In [100]:
!wget -q https://github.com/unicamp-dl/ia025_machado_livros/releases/download/v1.0/livros_com_nomes.zip
!unzip -o -q livros_com_nomes.zip
!wget -q https://github.com/unicamp-dl/ia025_machado_livros/releases/download/v1.0/livros_sem_nomes.zip
!unzip -o -q livros_sem_nomes.zip

Exemplo de abertura de um trecho de um dos livros:

In [ ]:
import os

DATA_PATH = os.path.join("livros_sem_nomes", "helena.txt")

with open(DATA_PATH, "r") as data_file:
    lines = [line for line in data_file]

full_data = ' '.join(lines)
full_data[1000:2000]

'o lado materno\n descendia de uma das mais distintas famílias paulistas. Ele próprio exercera dois empregos,\n havendose com habilidade e decoro, do que lhe adveio a carta de conselho e a estima dos\n homens públicos. Sem embargo do ardor político do tempo, não estava ligado a nenhum dos\n dois partidos, conservando em ambos preciosas amizades, que ali se acharam na ocasião de o\n dar à sepultura. Tinha, entretanto, tais ou quais idéias políticas, colhidas nas fronteiras con\n servadoras e liberais, justamente no ponto em que os dois domínios podem confundirse. Se\n nenhuma saudade partidária lhe deitou a última pá de terra, matrona houve, e não só uma, que\n viu ir a enterrar com ele a melhor página da sua mocidade.\n A família do conselheiro compunhase de duas pessoas: um filho, o [NOME], e uma\n irmã, [NOME]. Contava esta cinqüenta e poucos anos; era solteira; vivera sempre com o\n irmão, cuja casa dirigia desde o falecimento da cunhada. [NOME] tinha vinte e sete anos, e era\n form

In [ ]:
# TODO Separar em treino validação e teste

In [ ]:
"""TODO: Preparar o dataset. Lembre-se de usar o tokenizador e ajustar seus parâmetros."""

In [ ]:
"""TODO: implemente a classe do dataset, visualize se está funcionando"""

train_data = MyDataset(...)
val_data = ...
test_data ...

In [ ]:
""" TODO visualize se o dataloader faz sentido """
batch_size = 8
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)
sample = next(iter(train_loader))

## Model

In [ ]:
import torch.nn as nn

class TextClassifier(torch.nn.Module):
    """TODO: implementar o modelo classificador de texto com ajuste-fino de um BERT"""
    def __init__(self):
      raise NotImplementedError
      """TODO:
        Iniciliazar o BERT
        Inicializar MLP de classificação usando os embeddings finais do token [CLS]
      """

    def forward(self):
      """TODO
      """
      raise NotImplementedError


In [ ]:
# Verifica se há uma GPU disponível e define o dispositivo para GPU se possível, caso contrário, usa a CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
epochs = """?"""
lr = """TODO: experimento""""
criterion = """TODO CrossEntropy""""

optimizer = """TODO: AdamW?""""

model.to(device)

"""TODO: Implemente o loop de treinamento. Em cada época, calcule e imprima a loss/acurácia no dataset de validação, e escolha o melhor modelo pela validação""""

## Avaliação

In [ ]:
""" TODO: calcule métricas de classificação final no dataset de teste. Seria legal ver métricas por classe (livro)! """

## Exemplos


In [ ]:
""" TODO: selecione trechos aleatórios de teste e mostre alguns casos se o modelo está acertando """

## Discussão

## Disclaimer: Uso de IA